# XAI stability and faithfulness metrics 
-- RIS, ROS, RES, PGI, LAP, and CAI

In [ ]:
# Metrics for 
#   T-Explainer,
#   SHAP Explainer
#   LIME
#   Integrated Gradients
#   Input X Gradient
#   DeepLIFT
#   LRP

In [ ]:
!pip install xgboost
!pip install torch
!pip install captum
!pip install lime
!pip install shap

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
#!pip install -e OpenXAI

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import xgboost as xgb

import lime
import lime.lime_tabular

from captum.attr import IntegratedGradients
from captum.attr import InputXGradient
from captum.attr import DeepLift
from captum.attr import LRP

import shap
shap.initjs()

import warnings
from tqdm import tqdm

In [ ]:
#import openxai

# Data loaders
#from openxai.dataloader import return_loaders

# Perturbation methods required for the computation of the relative stability metrics
#from openxai.explainers.catalog.perturbation_methods import NormalPerturbation
#from openxai.explainers.catalog.perturbation_methods import NewDiscrete_NormalPerturbation


# classes from OpenXAI (Agarwal, Chirag, et al., 2022)
# we cloned theses classes here due to compatibility issues with OpenXAI
# we included in get_perturbed_inputs an optional generator in order to generate
# pseudorandom samplin

class BasePerturbation:
    '''
    Base Class for perturbation methods.
    '''
    
    def __init__(self, data_format):
        '''
        Initialize generic parameters for the perturbation method
        '''
        self.data_format = data_format
    
    def get_perturbed_inputs(self):
        '''
        This function implements the logic of the perturbation methods which will return perturbed samples.
        '''
        pass
    
class NormalPerturbation(BasePerturbation):
    def __init__(self, data_format, mean: int = 0, std_dev: float = 0.05, flip_percentage: float = 0.3):
        self.mean = mean
        self.std_dev = std_dev
        self.flip_percentage = flip_percentage

        super(NormalPerturbation, self).__init__(data_format)
        '''
        Initializes the marginal perturbation method where each column is sampled from marginal distributions 
        given per variable. dist_per_feature : vector of distribution generators 
        (tdist under torch.distributions).
        Note : These distributions are assumed to have zero mean since they get added to the original sample.
        '''
        pass

    def get_perturbed_inputs(self, original_sample: torch.FloatTensor, feature_mask: torch.BoolTensor,
                             num_samples: int, feature_metadata: list, max_distance: int = None, 
                             generator= None) -> torch.tensor:
        '''
        feature mask : this indicates the static features
        num_samples : number of perturbed samples.
        max_distance : the maximum distance between original sample and purturbed samples.
        generator: generator (torch.Generator, optional) – a pseudorandom number generator for sampling
        '''
        feature_type = feature_metadata
        assert len(feature_mask) == len(
            original_sample), "mask size == original sample in get_perturbed_inputs for {}".format(self.__class__)

        perturbed_cols = []
        continuous_features = torch.tensor([i == 'c' for i in feature_type])
        discrete_features = torch.tensor([i == 'd' for i in feature_type])

        # Processing continuous columns -- generator inclusion
        mean = self.mean
        std_dev = self.std_dev
        perturbations = torch.normal(
            mean, std_dev, [num_samples, len(feature_type)], generator=generator
        ) * continuous_features + original_sample

        # Processing discrete columns
        flip_percentage = self.flip_percentage
        p = torch.empty(num_samples, len(feature_type)).fill_(flip_percentage)
        perturbations = perturbations * (~discrete_features) + torch.abs(
            (perturbations * discrete_features) - (torch.bernoulli(p) * discrete_features))

        # keeping features static that are in top-K based on feature mask
        perturbed_samples = original_sample * feature_mask + perturbations * (~feature_mask)

        return perturbed_samples

In [5]:
# Perturbation class parameters
perturbation_mean= 0.0
perturbation_std= 0.01
perturbation_flip_percentage= 0.0001
    
perturbation= NormalPerturbation('tabular',
                                 mean=perturbation_mean,
                                 std_dev=perturbation_std,
                                 flip_percentage=perturbation_flip_percentage)

def generate_mask(explanation, top_k):
    mask_indices= torch.topk(explanation, top_k).indices
    mask= torch.zeros(explanation.shape) > 10
    for i in mask_indices:
        mask[i]= True
    return mask

# Quantitative Metrics -- Auxiliar Methods

In [ ]:
class MLPClassifierModel(nn.Module):
    """
    Create a custom PyTorch model that mimics the behavior of a scikit-learn MLPClassifier model
    Define the PyTorch Neural Network model (ReLU MLP-based)
    """
    def __init__(self, input_size, hidden_sizes, output_size):
        super(MLPClassifierModel, self).__init__()
        
        self.layers= nn.ModuleList([nn.Linear(input_size, hidden_sizes[0])])
        self.activations= [nn.ReLU()]
        
        for i in range(1, len(hidden_sizes)):
            self.layers.append(nn.Linear(hidden_sizes[i-1], hidden_sizes[i]))
            self.activations.append(nn.ReLU())
        
        self.output_layer= nn.Linear(hidden_sizes[-1], output_size)

        
    def forward(self, x):
        for layer, activation in zip(self.layers, self.activations):
            x= activation(layer(x))
        
        x= self.output_layer(x)
        
        return x

In [ ]:
def sklearn_to_pytorch_NN(skl_nn_model, input_size, output_size=1):
    """
    convert a scikit-learn NN model to a PyTorch NN model

    skl_nn_model is the scikit-learn Neural Net model
    input_size is the number of input features
    
    RETURN a PyTorch Neural Net model used to binary classifications
    binary classification -- one output neuron for the binary prediction
    """

    input_size= input_size
    hidden_sizes= skl_nn_model.hidden_layer_sizes

    nn_pytorch_model= MLPClassifierModel(input_size, hidden_sizes, output_size)

    # Transfer the weights from the scikit-learn model to the PyTorch model
    for i, layer in enumerate(nn_pytorch_model.layers):
        layer.weight.data= torch.tensor(skl_nn_model.coefs_[i].T, dtype=torch.float32)
        layer.bias.data= torch.tensor(skl_nn_model.intercepts_[i], dtype=torch.float32)

    nn_pytorch_model.output_layer.weight.data= torch.tensor(skl_nn_model.coefs_[-1].T, dtype=torch.float32)
    nn_pytorch_model.output_layer.bias.data= torch.tensor(skl_nn_model.intercepts_[-1], dtype=torch.float32)
    
    
    return nn_pytorch_model

In [ ]:
def distance_ordering(tensor_x, tensor_y, target_x):
    """
    RETURN tensor_x and tensor_y datsets (tensors) according to euclidean distance ordering from target_x
    """
    
    # Calculate Euclidean distances for each row
    distances= torch.norm((tensor_x - target_x), dim=1)

    # Sort the data tensor based on distances
    sorted_indices= torch.argsort(distances)
    
    sorted_x= tensor_x[sorted_indices]
    sorted_y= tensor_y[sorted_indices]
    
    return sorted_x, sorted_y

In [ ]:
def remove_tensor_row_by_indexset(dataset, index_to_remove):
    """
    Remove a set of rows in a tensor dataset by index

    dataset is a n elements dataset
    index_to_remove is a m elements tensor with the indexes to remove
    
    RETURN a subset form dataset without the index_to_remove instances
    """

    # Ensure indices are unique and sorted (if necessary)
    index_to_remove= torch.unique(index_to_remove)
    
    # Generate a mask of rows to keep
    mask= torch.ones(dataset.size(0), dtype=torch.bool)
    mask[index_to_remove]= False  # Set indices to remove as False

    # Apply the mask to filter the dataset
    subset= dataset[mask]

    return subset

In [ ]:
def get_subsets(x, x_class, dataset, dataset_class, n_elements, option:int=0):
    """ 
    Obtain a subset from a dataset with at least n_elements.

    x is a tensor instance
    x_class is a tensor with the class of x
    dataset is a m elements tensor dataset
    dataset_class is a m elements tensor with the predicted 
    n_elements is an integer indicating the size of the subset
    
    RETURN two tensor subsets (from dataset and dataset_class) with 
           option 1 - n_elements ordered first by class (same from x) and then by distance from x
           option 0 - n_elements ordered by class (same from x) and filled (if necessary) with x and x_class
    
    option 0 gives us y' = y for all x' and option 1 relaxes such a restriction
    """
    
    data_size= dataset.shape[0]
    
    if (data_size< n_elements):
        raise ValueError("Data size must be greater than n_elements!")

    # Option 1: Order by distance and filter by class
    if option:
        # Order dataset by distance from `x`
        dataset_order, dataset_class_order= distance_ordering(dataset, dataset_class, x.unsqueeze(0))

        # Get indices of elements matching `x_class`
        ind_same_class= (dataset_class_order == x_class).nonzero(as_tuple=True)[0]
        subset= dataset_order[ind_same_class][:n_elements]
        subset_class= dataset_class_order[ind_same_class][:n_elements]

    # Option 0: Filter by class only
    else:
        ind_same_class= (dataset_class == x_class).nonzero(as_tuple=True)[0]
        subset= dataset[ind_same_class][:n_elements]
        subset_class= dataset_class[ind_same_class][:n_elements]

    # If subset size is less than required, fill remaining
    if subset.shape[0] < n_elements:
        remaining= n_elements - subset.shape[0]

        if option:
            # Remove already selected indices and pick the next closest points
            unselected_mask= torch.ones(dataset_order.shape[0], dtype=torch.bool)
            unselected_mask[ind_same_class]= False
            additional= dataset_order[unselected_mask][:remaining]
            additional_class= dataset_class_order[unselected_mask][:remaining]

            subset= torch.cat((subset, additional))
            subset_class= torch.cat((subset_class, additional_class))
        else:
            # Fill with copies of `x` and `x_class`
            fill_x= x.repeat((remaining, 1))
            fill_x_class= x_class.repeat(remaining)

            subset= torch.cat((subset, fill_x))
            subset_class= torch.cat((subset_class, fill_x_class))

    return subset, subset_class

In [ ]:
# clip values near to zero in v replacing by eps

# v is a single value (float) or a numpy.ndarray with (n,) shape
# eps is a small number of tolerance limiting what is a small value

# RETURN v clipped

def clip_small_values(v, eps=1e-6):
    
    if isinstance(v, np.ndarray):
        # Vectorized clipping for arrays
        v_clipped= np.where((v < 0) & (np.abs(v) < eps), -eps, v)
        v_clipped= np.where((v > 0) & (v < eps), eps, v_clipped)
    else:
        # Scalar clipping
        if (v < 0 and abs(v) < eps):
            v_clipped = -eps
        elif (v > 0 and v < eps):
            v_clipped = eps
        else:
            v_clipped = v

    return v_clipped

In [ ]:
# returns the square of the difference of any two quantities v1 and v2.

def square_difference(v1, v2):
    
    # arrays can be flattened, so long as ordering is preserved
    v1_flat= np.asarray(v1).flatten()
    v2_flat= np.asarray(v2).flatten()
    
    dif_flat= (v1_flat - v2_flat)
    
    return np.power(dif_flat, 2)

In [ ]:
# returns the Lp norm of the difference between v1 and v2.
# normalizes the difference between v1 and v2 by v1 (adapted; Agarwal, Chirag, et al., 2022)

def lp_norm_dif(v1, v2, p_norm=2, eps=1e-6, norm:bool=True):
    
    # arrays can be flattened, so long as ordering is preserved
    v1_flat= np.asarray(v1).flatten()
    v2_flat= np.asarray(v2).flatten()
    
    dif_flat= (v1_flat - v2_flat)
    
    if (norm==True):
        v1_flat= clip_small_values(v1_flat, eps)
        
        dif_flat= np.divide(dif_flat, v1_flat, out=np.zeros_like(v1_flat), where=v1_flat!=0)

    return np.linalg.norm(dif_flat, ord=p_norm)

In [ ]:
def ris_measure(x_data, x_pert, exp_data, exp_pert, p_norm=2, eps=1e-6):
    """ compute norm between predictions per perturbation - RIS """
    
    x_dif_norm= lp_norm_dif(x_data, x_pert, p_norm=p_norm, eps=eps, norm=True)
    #x_dif_norm= np.clip(x_dif_norm, eps, None)
    x_dif_norm= clip_small_values(x_dif_norm, eps)
    
    exp_dif_norm= lp_norm_dif(exp_data, exp_pert, p_norm=p_norm, eps=eps, norm=True)
    
    stability_measure= np.divide(exp_dif_norm, x_dif_norm, where=x_dif_norm!=0)
    
    return stability_measure

In [ ]:
def ros_measure(fx_data, fx_pert, exp_data, exp_pert, p_norm=2, eps=1e-6):
    """
    compute norm between representations - ROS
    x_data and x_pert must to be pd.DataFrame row individual instances with column names
    """
    
    fx_dif_norm= lp_norm_dif(fx_data, fx_pert, p_norm=p_norm, eps=eps, norm=True)
    #fx_dif_norm= np.clip(fx_dif_norm, eps, None)
    fx_dif_norm= clip_small_values(fx_dif_norm, eps)
    
    exp_dif_norm= lp_norm_dif(exp_data, exp_pert, p_norm=p_norm, eps=eps, norm=True)

    stability_measure= np.divide(exp_dif_norm, fx_dif_norm, where=fx_dif_norm!=0)
    
    return stability_measure

In [ ]:
def lime_exp_in_data_order(lime_exp, num_fts):
    """
    bring explanations into data order (since LIME automatically orders according to highest importance)
    """
    
    exp= np.zeros(num_fts)

    for k, v in lime_exp.local_exp[1]:
        exp[k]= v

    return exp

In [ ]:
def get_x_explanations(model, data, labels, target_x, target_y, descriptor, cat_fts=[], 
                       nn_pytorch_model=None, is_model_NN:bool=False, 
                       retrained=False, ohe_model=None):
    """
    Generate feature importance explanations for a single data instance.
    Args:
        model: A trained machine learning model (sklearn or XGBoost) -- binary classifier.
        data: DataFrame of training data.
        labels: DataFrame of training labels.
        target_x: DataFrame with the instance to explain.
        target_y: The true label of the instance to explain.
        descriptor: Dictionary defining parameters.
        cat_fts: List of categorical features (if any).
        nn_pytorch_model: PyTorch model (for gradient-based methods).
        is_model_NN: Whether the model is a neural network.
        retrained: Whether the model is retrained.
        ohe_model: Model trainded on the one-hot encoded categorical data (if applicable).
    Returns:
        A dictionary containing feature importance explanations from explainers
            T-Explainer
            SHAP
            LIME
            Integrated Gradients
            Input X Gradient
            DeepLIFT
            LRP
    """
    
    # ------------------------------------ x_data explanation
    t_x_exp= torch.zeros(data.shape[1])
    # """
    # data point explanation -- T-Exp
    if (retrained):    # Categorical case 
        t_x_exp, t_x_ft, t_x_sh= categorical_taylor_explainer(ohe_model, num_ohe_train_data, labels, 
                                          target_x, target_y, cat_cols=cat_fts,
                                          h_min=descriptor['h_min'], h_max=descriptor['h_max'],
                                          eps=descriptor['jacobian_eps'], max_itr=descriptor['max_itr'],
                                          finite_diff_version=descriptor['finite_diff_version'], 
                                          delta=descriptor['ohe_delta'], e_x=descriptor['e_x'], 
                                          angle=descriptor['angle'], retrained_ohe_model=True, 
                                          verbose=False)
    else:
        t_x_exp, t_x_ft, t_x_sh= taylor_explainer(model, data, labels, target_x, target_y,
                                          h_min=descriptor['h_min'], h_max=descriptor['h_max'],
                                          eps=descriptor['jacobian_eps'], max_itr=descriptor['max_itr'],
                                          finite_diff_version=descriptor['finite_diff_version'], 
                                          e_x=descriptor['e_x'], angle=descriptor['angle'],
                                          verbose=False)
    t_x_exp= torch.from_numpy(t_x_exp)
    # """

    # data point explanation -- SHAP
    if isinstance(model, xgb.XGBModel):
        shap_exp_gen= shap.TreeExplainer(model, data, model_output='probability')
    elif hasattr(model, 'predict'):
        shap_exp_gen= shap.Explainer(model.predict, data)
    else:
        shap_exp_gen= shap.Explainer(model.predict_proba, data)

    shap_x_exp= shap_exp_gen(target_x)
    shap_x_exp= (torch.from_numpy(shap_x_exp.values)).squeeze()


    # data point explanation -- LIME
    lime_exp_gen= lime.lime_tabular.LimeTabularExplainer(
        training_data=np.asarray(data), 
        feature_names=np.asarray(data.columns), 
        training_labels=labels.values.ravel().astype(int), 
        class_names=np.asarray([0,1]), 
        mode='classification', 
        discretize_continuous=False, 
        verbose=False
    )

    # ignore temporarily warnings related to feature names
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="X does not have valid feature names")
        lime_scores= lime_exp_gen.explain_instance(
            data_row=np.asarray(target_x)[0], 
            predict_fn=model.predict_proba, 
            num_features=data.shape[1]
        )

        lime_x_exp= torch.from_numpy(lime_exp_in_data_order(lime_scores, data.shape[1]))
    # reset the warning settings
    warnings.resetwarnings()
    
    
    # data point explanation -- Gradient-based methods
    if (is_model_NN==True):
        x_data_tensor= torch.tensor(np.asarray(target_x), dtype=torch.float32, requires_grad=True)
        # ignore non-important warnings temporarily
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Setting")
        
            itGd= IntegratedGradients(nn_pytorch_model)
            itGd_x_exp= (itGd.attribute(x_data_tensor)).squeeze().detach()

            iXGd= InputXGradient(nn_pytorch_model)
            iXGd_x_exp= (iXGd.attribute(x_data_tensor.unsqueeze(0))).squeeze().detach()

            dLif= DeepLift(nn_pytorch_model)
            dLif_x_exp= (dLif.attribute(x_data_tensor.unsqueeze(0))).squeeze().detach()

            lwrp= LRP(nn_pytorch_model)
            lwrp_x_exp= (lwrp.attribute(x_data_tensor.unsqueeze(0))).squeeze().detach()
        # reset the warning settings
        warnings.resetwarnings()
    else:
        itGd_x_exp= torch.zeros(data.shape[1])
        iXGd_x_exp= torch.zeros(data.shape[1])
        dLif_x_exp= torch.zeros(data.shape[1])
        lwrp_x_exp= torch.zeros(data.shape[1])

    importances= {
        't_exp': t_x_exp, 'lime': lime_x_exp, 'shap': shap_x_exp,
        'itGd': itGd_x_exp, 'iXGd': iXGd_x_exp, 'dLif': dLif_x_exp, 'lwrp': lwrp_x_exp, 
    }
        
    return importances


# Metric -- Relative Input/Output Stability -- RIS / ROS

In [ ]:
def relative_stability(model, data, labels, perturbation, descriptor, cat_fts=[], is_model_NN:bool=False, 
                       generator=None):
    """
    Relative Input/Output Stability (RIS/ROS) Metric Computation
    This function evaluates the stability of feature attribution explanations 
    by computing the Relative Input Stability (RIS) and Relative Output Stability (ROS) metrics.
    Args:
        model: A trained machine learning model (sklearn or XGBoost) -- binary classifier.
        data: DataFrame of training data.
        labels: DataFrame of training labels.
        perturbation (OpenXAI object): Object used to generate perturbed samples.
        descriptor: Dictionary defining parameters.
        cat_fts: (list, optional): List of categorical feature indices. If empty, all features are treated as numeric.
        is_model_NN (bool, optional): Whether the model is an 'MLPClassifier' (Neural Network). Defaults to False.
        generator (torch.Generator, optional): Pseudorandom number generator for sampling.
    Returns:
        dict: Dictionary containing RIS and ROS metrics for different explanation methods 
            T-Explainer
            SHAP
            LIME
            Integrated Gradients
            Input X Gradient
            DeepLIFT
            LRP
    """
    
    t_ris_max_ratios= []
    shap_ris_max_ratios= []
    lime_ris_max_ratios= []
    
    t_ris_mean_ratios= []
    shap_ris_mean_ratios= []
    lime_ris_mean_ratios= []
    
    t_ros_max_ratios= []
    shap_ros_max_ratios= []
    lime_ros_max_ratios= []
    
    t_ros_mean_ratios= []
    shap_ros_mean_ratios= []
    lime_ros_mean_ratios= []

    if (is_model_NN==True):
        # convert a scikit-learn NN model to a PyTorch NN model used in captum
        nn_pytorch_model= sklearn_to_pytorch_NN(model, data.shape[1])
            
        itGd_ris_max_ratios= []
        iXGd_ris_max_ratios= []
        dLif_ris_max_ratios= []
        lwrp_ris_max_ratios= []

        itGd_ris_mean_ratios= []
        iXGd_ris_mean_ratios= []
        dLif_ris_mean_ratios= []
        lwrp_ris_mean_ratios= []

        itGd_ros_max_ratios= []
        iXGd_ros_max_ratios= []
        dLif_ros_max_ratios= []
        lwrp_ros_max_ratios= []

        itGd_ros_mean_ratios= []
        iXGd_ros_mean_ratios= []
        dLif_ros_mean_ratios= []
        lwrp_ros_mean_ratios= []
    else:
        nn_pytorch_model= None
        
    
    ohe_model= None
    retrained= False
    if (np.asarray(cat_fts).shape[0]> 0):
        # T-Explainer requires a model retraining only for categorical cases
        num_ohe_train_data= ohe_cat_to_numerical_simulator(train_data, cat_fts, 
                                                           delta=descriptor['ohe_delta'], 
                                                           rand_seed=True)
        ohe_model= clone(model)
        ohe_model.fit(num_ohe_train_data, labels_train.values.ravel())
        retrained= True

    
    data_size= data.shape[0]
    
    for i_data in tqdm(range(data_size)):
        # i_data and its label as pd.DataFrames
        target_x= pd.DataFrame(data=[data.iloc[i_data,:]], columns=data.columns)
        target_y= pd.DataFrame(data=[labels.iloc[i_data]], columns=labels.columns)

        # i_data and its label as tensors
        x_data= torch.tensor(np.asarray(target_x), dtype=torch.float64)
        y_data= torch.tensor(np.asarray(target_y), dtype=torch.float64)

        # data point prediction
        fx_data= ML(model, target_x)
        y_pred= torch.from_numpy(model.predict(target_x).astype(int))

        # ------------------------------------ get x_data explanation
        x_exps= get_x_explanations(model, data, labels, target_x, target_y, descriptor, cat_fts, 
                                   nn_pytorch_model, is_model_NN, retrained, ohe_model)

        t_x_exp= x_exps['t_exp']
        lime_x_exp= x_exps['lime']
        shap_x_exp= x_exps['shap']
        itGd_x_exp= x_exps['itGd']
        iXGd_x_exp= x_exps['iXGd']
        dLif_x_exp= x_exps['dLif']
        lwrp_x_exp= x_exps['lwrp']

        
        # ------------------------------------ x_data perturbation
        # data point perturbation
        x_pert_samples= perturbation.get_perturbed_inputs(
            original_sample=x_data.reshape(-1), feature_mask=descriptor['mask'],
            num_samples=descriptor['num_samples'], max_distance=descriptor['pert_max_distance'],
            feature_metadata=descriptor['feature_metadata'], generator=generator
        )

        # --- take the closest num_perts points to x_data that have the same predicted class label to x_data
        y_pert_preds= torch.from_numpy(
            model.predict(pd.DataFrame(data=x_pert_samples.numpy(), columns=data.columns)).astype(int)
        )
        
        # get only the first num_perts points ordered by class and distance from x_data
        x_pert_samples, y_pert_preds= get_subsets(
            x_data.reshape(-1), y_pred, x_pert_samples, y_pert_preds, descriptor['num_perts']
        )
        
        
        # ------------------------------------ explain each x_data perturbation
        t_exp_pert_samples= torch.zeros_like(x_pert_samples)
        shap_exp_pert_samples= torch.zeros_like(x_pert_samples)
        lime_exp_pert_samples= torch.zeros_like(x_pert_samples)
        
        if (is_model_NN==True):
            itGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
            iXGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
            dLif_exp_pert_samples= torch.zeros_like(x_pert_samples)
            lwrp_exp_pert_samples= torch.zeros_like(x_pert_samples)
        
        
        t_x_ris_ratios= []
        shap_x_ris_ratios= []
        lime_x_ris_ratios= []
        
        t_x_ros_ratios= []
        shap_x_ros_ratios= []
        lime_x_ros_ratios= []
        
        if (is_model_NN==True):
            itGd_x_ris_ratios= []
            iXGd_x_ris_ratios= []
            dLif_x_ris_ratios= []
            lwrp_x_ris_ratios= []

            itGd_x_ros_ratios= []
            iXGd_x_ros_ratios= []
            dLif_x_ros_ratios= []
            lwrp_x_ros_ratios= []
        
        # For each perturbation, calculate the explanation
        for i, x_pert in enumerate(x_pert_samples):
            
            df_x_pert= pd.DataFrame(data=[x_pert.numpy()], columns=data.columns)
            df_y_pert= pd.DataFrame(data=[np.int64(y_pert_preds[i])], columns=labels.columns)
            fx_pert= ML(model, df_x_pert)
            
            # ------------------------------------ x_pert explanation
            x_exp_pert= get_x_explanations(model, data, labels, df_x_pert, df_y_pert, descriptor, cat_fts, 
                                           nn_pytorch_model, is_model_NN, retrained, ohe_model)

            t_exp_pert_samples[i, :]= x_exp_pert['t_exp']
            lime_exp_pert_samples[i, :]= x_exp_pert['lime']
            shap_exp_pert_samples[i, :]= x_exp_pert['shap']
            itGd_exp_pert_samples[i, :]= x_exp_pert['itGd']
            iXGd_exp_pert_samples[i, :]= x_exp_pert['iXGd']
            dLif_exp_pert_samples[i, :]= x_exp_pert['dLif']
            lwrp_exp_pert_samples[i, :]= x_exp_pert['lwrp']
        
    
            # ------------------------------------ get stability for each explanator and x_data perturbation
            # RIS
            t_ris_measure= ris_measure(x_data, x_pert, 
                                       t_x_exp, t_exp_pert_samples[i], 
                                       p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            shap_ris_measure= ris_measure(x_data, x_pert, 
                                          shap_x_exp, shap_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            lime_ris_measure= ris_measure(x_data, x_pert, 
                                          lime_x_exp, lime_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            if (is_model_NN==True):
                itGd_ris_measure= ris_measure(x_data, x_pert, 
                                             itGd_x_exp, itGd_exp_pert_samples[i], 
                                             p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                iXGd_ris_measure= ris_measure(x_data, x_pert, 
                                              iXGd_x_exp, iXGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                dLif_ris_measure= ris_measure(x_data, x_pert, 
                                              dLif_x_exp, dLif_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                lwrp_ris_measure= ris_measure(x_data, x_pert, 
                                              lwrp_x_exp, lwrp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
            # ROS
            t_ros_measure= ros_measure(fx_data, fx_pert, 
                                       t_x_exp, t_exp_pert_samples[i], 
                                       p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            shap_ros_measure= ros_measure(fx_data, fx_pert, 
                                          shap_x_exp, shap_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            lime_ros_measure= ros_measure(fx_data, fx_pert, 
                                          lime_x_exp, lime_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            if (is_model_NN==True):
                itGd_ros_measure= ros_measure(fx_data, fx_pert, 
                                              itGd_x_exp, itGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                iXGd_ros_measure= ros_measure(fx_data, fx_pert, 
                                              iXGd_x_exp, iXGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                dLif_ros_measure= ros_measure(fx_data, fx_pert, 
                                              dLif_x_exp, dLif_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                lwrp_ros_measure= ros_measure(fx_data, fx_pert, 
                                              lwrp_x_exp, lwrp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
        
            
            # --- stability measures for each x_data perturbation --- one processing cicle
            # RIS
            t_x_ris_ratios.append(t_ris_measure)
            shap_x_ris_ratios.append(shap_ris_measure)
            lime_x_ris_ratios.append(lime_ris_measure)
            
            # ROS
            t_x_ros_ratios.append(t_ros_measure)
            shap_x_ros_ratios.append(shap_ros_measure)
            lime_x_ros_ratios.append(lime_ros_measure)
            
            if (is_model_NN==True):
                itGd_x_ris_ratios.append(itGd_ris_measure)
                iXGd_x_ris_ratios.append(iXGd_ris_measure)
                dLif_x_ris_ratios.append(dLif_ris_measure)
                lwrp_x_ris_ratios.append(lwrp_ris_measure)

                itGd_x_ros_ratios.append(itGd_ros_measure)
                iXGd_x_ros_ratios.append(iXGd_ros_measure)
                dLif_x_ros_ratios.append(dLif_ros_measure)
                lwrp_x_ros_ratios.append(lwrp_ros_measure)
        
        # --- append only the max/mean value related to each x_data processed
        # max values
        t_ris_max_ratios.append(t_x_ris_ratios[np.argmax(t_x_ris_ratios)])
        shap_ris_max_ratios.append(shap_x_ris_ratios[np.argmax(shap_x_ris_ratios)])
        lime_ris_max_ratios.append(lime_x_ris_ratios[np.argmax(lime_x_ris_ratios)])

        t_ros_max_ratios.append(t_x_ros_ratios[np.argmax(t_x_ros_ratios)])
        shap_ros_max_ratios.append(shap_x_ros_ratios[np.argmax(shap_x_ros_ratios)])
        lime_ros_max_ratios.append(lime_x_ros_ratios[np.argmax(lime_x_ros_ratios)])
        
        # mean values
        t_ris_mean_ratios.append(np.mean(t_x_ris_ratios))
        shap_ris_mean_ratios.append(np.mean(shap_x_ris_ratios))
        lime_ris_mean_ratios.append(np.mean(lime_x_ris_ratios))

        t_ros_mean_ratios.append(np.mean(t_x_ros_ratios))
        shap_ros_mean_ratios.append(np.mean(shap_x_ros_ratios))
        lime_ros_mean_ratios.append(np.mean(lime_x_ros_ratios))
        
        if (is_model_NN==True):
            # max values
            itGd_ris_max_ratios.append(itGd_x_ris_ratios[np.argmax(itGd_x_ris_ratios)])
            iXGd_ris_max_ratios.append(iXGd_x_ris_ratios[np.argmax(iXGd_x_ris_ratios)])
            dLif_ris_max_ratios.append(dLif_x_ris_ratios[np.argmax(dLif_x_ris_ratios)])
            lwrp_ris_max_ratios.append(lwrp_x_ris_ratios[np.argmax(lwrp_x_ris_ratios)])

            itGd_ros_max_ratios.append(itGd_x_ros_ratios[np.argmax(itGd_x_ros_ratios)])
            iXGd_ros_max_ratios.append(iXGd_x_ros_ratios[np.argmax(iXGd_x_ros_ratios)])
            dLif_ros_max_ratios.append(dLif_x_ros_ratios[np.argmax(dLif_x_ros_ratios)])
            lwrp_ros_max_ratios.append(lwrp_x_ros_ratios[np.argmax(lwrp_x_ros_ratios)])

            # mean values
            itGd_ris_mean_ratios.append(np.mean(itGd_x_ris_ratios))
            iXGd_ris_mean_ratios.append(np.mean(iXGd_x_ris_ratios))
            dLif_ris_mean_ratios.append(np.mean(dLif_x_ris_ratios))
            lwrp_ris_mean_ratios.append(np.mean(lwrp_x_ris_ratios))

            itGd_ros_mean_ratios.append(np.mean(itGd_x_ros_ratios))
            iXGd_ros_mean_ratios.append(np.mean(iXGd_x_ros_ratios))
            dLif_ros_mean_ratios.append(np.mean(dLif_x_ros_ratios))
            lwrp_ros_mean_ratios.append(np.mean(lwrp_x_ros_ratios))
           
    # ------------------------------------ RETURN ratios considering all data processed
    t_ris_max = t_ris_max_ratios[np.argmax(t_ris_max_ratios)]
    shap_ris_max= shap_ris_max_ratios[np.argmax(shap_ris_max_ratios)]
    lime_ris_max= lime_ris_max_ratios[np.argmax(lime_ris_max_ratios)]
    
    t_ris_max_std = np.std(t_ris_max_ratios)
    shap_ris_max_std= np.std(shap_ris_max_ratios)
    lime_ris_max_std= np.std(lime_ris_max_ratios)
    
    t_ros_max = t_ros_max_ratios[np.argmax(t_ros_max_ratios)]
    shap_ros_max= shap_ros_max_ratios[np.argmax(shap_ros_max_ratios)]
    lime_ros_max= lime_ros_max_ratios[np.argmax(lime_ros_max_ratios)]
    
    t_ros_max_std = np.std(t_ros_max_ratios)
    shap_ros_max_std= np.std(shap_ros_max_ratios)
    lime_ros_max_std= np.std(lime_ros_max_ratios)
    
        
    t_ris_mean = np.mean(t_ris_mean_ratios)
    shap_ris_mean= np.mean(shap_ris_mean_ratios)
    lime_ris_mean= np.mean(lime_ris_mean_ratios)
    
    t_ris_mean_std = np.std(t_ris_mean_ratios)
    shap_ris_mean_std= np.std(shap_ris_mean_ratios)
    lime_ris_mean_std= np.std(lime_ris_mean_ratios)
    
    t_ros_mean = np.mean(t_ros_mean_ratios)
    shap_ros_mean= np.mean(shap_ros_mean_ratios)
    lime_ros_mean= np.mean(lime_ros_mean_ratios)
    
    t_ros_mean_std = np.std(t_ros_mean_ratios)
    shap_ros_mean_std= np.std(shap_ros_mean_ratios)
    lime_ros_mean_std= np.std(lime_ros_mean_ratios)
    
    if (is_model_NN==True):
        itGd_ris_max= itGd_ris_max_ratios[np.argmax(itGd_ris_max_ratios)]
        iXGd_ris_max= iXGd_ris_max_ratios[np.argmax(iXGd_ris_max_ratios)]
        dLif_ris_max= dLif_ris_max_ratios[np.argmax(dLif_ris_max_ratios)]
        lwrp_ris_max= lwrp_ris_max_ratios[np.argmax(lwrp_ris_max_ratios)]

        itGd_ris_max_std= np.std(itGd_ris_max_ratios)
        iXGd_ris_max_std= np.std(iXGd_ris_max_ratios)
        dLif_ris_max_std= np.std(dLif_ris_max_ratios)
        lwrp_ris_max_std= np.std(lwrp_ris_max_ratios)

        itGd_ros_max= itGd_ros_max_ratios[np.argmax(itGd_ros_max_ratios)]
        iXGd_ros_max= iXGd_ros_max_ratios[np.argmax(iXGd_ros_max_ratios)]
        dLif_ros_max= dLif_ros_max_ratios[np.argmax(dLif_ros_max_ratios)]
        lwrp_ros_max= lwrp_ros_max_ratios[np.argmax(lwrp_ros_max_ratios)]

        itGd_ros_max_std= np.std(itGd_ros_max_ratios)
        iXGd_ros_max_std= np.std(iXGd_ros_max_ratios)
        dLif_ros_max_std= np.std(dLif_ros_max_ratios)
        lwrp_ros_max_std= np.std(lwrp_ros_max_ratios)
        
        
        itGd_ris_mean= np.mean(itGd_ris_mean_ratios)
        iXGd_ris_mean= np.mean(iXGd_ris_mean_ratios)
        dLif_ris_mean= np.mean(dLif_ris_mean_ratios)
        lwrp_ris_mean= np.mean(lwrp_ris_mean_ratios)

        itGd_ris_mean_std= np.std(itGd_ris_mean_ratios)
        iXGd_ris_mean_std= np.std(iXGd_ris_mean_ratios)
        dLif_ris_mean_std= np.std(dLif_ris_mean_ratios)
        lwrp_ris_mean_std= np.std(lwrp_ris_mean_ratios)

        itGd_ros_mean= np.mean(itGd_ros_mean_ratios)
        iXGd_ros_mean= np.mean(iXGd_ros_mean_ratios)
        dLif_ros_mean= np.mean(dLif_ros_mean_ratios)
        lwrp_ros_mean= np.mean(lwrp_ros_mean_ratios)

        itGd_ros_mean_std= np.std(itGd_ros_mean_ratios)
        iXGd_ros_mean_std= np.std(iXGd_ros_mean_ratios)
        dLif_ros_mean_std= np.std(dLif_ros_mean_ratios)
        lwrp_ros_mean_std= np.std(lwrp_ros_mean_ratios)
        
        
    results= {
        't_exp_ris_max': t_ris_max, 'std(t_exp_ris_max)': t_ris_max_std,
        'shap_ris_max': shap_ris_max, 'std(shap_ris_max)': shap_ris_max_std,
        'lime_ris_max': lime_ris_max, 'std(lime_ris_max)': lime_ris_max_std,
        't_exp_ris_mean': t_ris_mean, 'std(t_exp_ris_mean)': t_ris_mean_std,
        'shap_ris_mean': shap_ris_mean, 'std(shap_ris_mean)': shap_ris_mean_std,
        'lime_ris_mean': lime_ris_mean, 'std(lime_ris_mean)': lime_ris_mean_std,
        't_exp_ros_max': t_ros_max, 'std(t_exp_ros_max)': t_ros_max_std,
        'shap_ros_max': shap_ros_max, 'std(shap_ros_max)': shap_ros_max_std,
        'lime_ros_max': lime_ros_max, 'std(lime_ros_max)': lime_ros_max_std,
        't_exp_ros_mean': t_ros_mean, 'std(t_exp_ros_mean)': t_ros_mean_std,
        'shap_ros_mean': shap_ros_mean, 'std(shap_ros_mean)': shap_ros_mean_std,
        'lime_ros_mean': lime_ros_mean, 'std(lime_ros_mean)': lime_ros_mean_std
    }

    if (is_model_NN==True):
        results_grad= {
            'itGd_ris_max': itGd_ris_max, 'std(itGd_ris_max)': itGd_ris_max_std,
            'iXGd_ris_max': iXGd_ris_max, 'std(iXGd_ris_max)': iXGd_ris_max_std,
            'dLif_ris_max': dLif_ris_max, 'std(dLif_ris_max)': dLif_ris_max_std,
            'lwrp_ris_max': lwrp_ris_max, 'std(lwrp_ris_max)': lwrp_ris_max_std,
            'itGd_ris_mean': itGd_ris_mean, 'std(itGd_ris_mean)': itGd_ris_mean_std,
            'iXGd_ris_mean': iXGd_ris_mean, 'std(iXGd_ris_mean)': iXGd_ris_mean_std,
            'dLif_ris_mean': dLif_ris_mean, 'std(dLif_ris_mean)': dLif_ris_mean_std,
            'lwrp_ris_mean': lwrp_ris_mean, 'std(lwrp_ris_mean)': lwrp_ris_mean_std,
            'itGd_ros_max': itGd_ros_max, 'std(itGd_ros_max)': itGd_ros_max_std,
            'iXGd_ros_max': iXGd_ros_max, 'std(iXGd_ros_max)': iXGd_ros_max_std,
            'dLif_ros_max': dLif_ros_max, 'std(dLif_ros_max)': dLif_ros_max_std,
            'lwrp_ros_max': lwrp_ros_max, 'std(lwrp_ros_max)': lwrp_ros_max_std,
            'itGd_ros_mean': itGd_ros_mean, 'std(itGd_ros_mean)': itGd_ros_mean_std,
            'iXGd_ros_mean': iXGd_ros_mean, 'std(iXGd_ros_mean)': iXGd_ros_mean_std,
            'dLif_ros_mean': dLif_ros_mean, 'std(dLif_ros_mean)': dLif_ros_mean_std,
            'lwrp_ros_mean': lwrp_ros_mean, 'std(lwrp_ros_mean)': lwrp_ros_mean_std 
        }
        results.update(results_grad)

    results= {
        key: [float(val) for val in value] if isinstance(value, list) else float(value) for key, value in results.items()
    }

    # the max/mean stability ratios
    return results

# Metric -- Run Explanation Stability -- RES

In [ ]:
def run_stability(model, data, labels, descriptor, cat_fts=[], is_model_NN:bool=False):
    """
    Computes explanation stability over multiple runs, returning the highest instability for each method.
    Args:
        model: A trained machine learning model (sklearn or XGBoost) -- binary classifier.
        data: DataFrame of training data.
        labels: DataFrame of training labels.
        descriptor: Dictionary defining parameters.
        cat_fts: (list, optional): List of categorical feature indices. If empty, all features are treated as numeric.
        is_model_NN (bool, optional): Whether the model is an 'MLPClassifier' (Neural Network). Defaults to False.
    Returns:
        Dictionary of max instability values for each explanation method.
    """
    
    t_stability_ratios= []
    shap_stability_ratios= []
    lime_stability_ratios= []

    nn_pytorch_model= None
    if (is_model_NN==True):
        # convert a scikit-learn NN model to a PyTorch NN model used in captum
        nn_pytorch_model= sklearn_to_pytorch_NN(model, data.shape[1])
            
        itGd_stability_ratios= []
        iXGd_stability_ratios= []
        dLif_stability_ratios= []
        lwrp_stability_ratios= []
        
        
    retrained= False
    ohe_model= None
    if (np.asarray(cat_fts).shape[0]> 0):
        # T-Explainer requires a model retraining only for categorical cases
        num_ohe_train_data= ohe_cat_to_numerical_simulator(train_data, cat_fts, 
                                                           delta=descriptor['ohe_delta'], 
                                                           rand_seed=True)
        ohe_model= clone(model)
        ohe_model.fit(num_ohe_train_data, labels_train.values.ravel())
        retrained= True
        
    
    runs= int(descriptor['num_runs'])
    data_size= data.shape[0]
    
    for i_data in tqdm(range(data_size)):

        # x_data and its label as pd.DataFrame
        target_x= pd.DataFrame(data=[data.iloc[i_data,:]], columns=data.columns)
        target_y= pd.DataFrame(data=[labels.iloc[i_data]], columns=labels.columns)

        t_x_exps= []
        shap_x_exps= []
        lime_x_exps= []
        
        if (is_model_NN==True):
            itGd_x_exps= []
            iXGd_x_exps= []
            dLif_x_exps= []
            lwrp_x_exps= []
                               
        for i in range(runs):
            
            # ------------------------------------ n runs x_data explanation
            x_exps= get_x_explanations(model, data, labels, target_x, target_y, descriptor, cat_fts, 
                                   nn_pytorch_model, is_model_NN, retrained, ohe_model)

            t_x_exp= x_exps['t_exp']
            lime_x_exp= x_exps['lime']
            shap_x_exp= x_exps['shap']
            itGd_x_exp= x_exps['itGd']
            iXGd_x_exp= x_exps['iXGd']
            dLif_x_exp= x_exps['dLif']
            lwrp_x_exp= x_exps['lwrp']
        
            
            # get the explanation of each method to each run            
            t_x_exps.append(t_x_exp)
            shap_x_exps.append(shap_x_exp.squeeze())
            lime_x_exps.append(lime_x_exp)
            
            if (is_model_NN==True):
                itGd_x_exps.append(itGd_x_exp.numpy())
                iXGd_x_exps.append(iXGd_x_exp.numpy())
                dLif_x_exps.append(dLif_x_exp.numpy())
                lwrp_x_exps.append(lwrp_x_exp.numpy())
                
                
        t_x_exps= np.asarray(t_x_exps)
        shap_x_exps= np.asarray(shap_x_exps)
        lime_x_exps= np.asarray(lime_x_exps)
            
        t_x_exps_mean= np.mean(t_x_exps, axis=0)
        shap_x_exps_mean= np.mean(shap_x_exps, axis=0)
        lime_x_exps_mean= np.mean(lime_x_exps, axis=0)
        
        t_x_exp_ratios= []
        shap_x_exp_ratios= []
        lime_x_exp_ratios= []
        
        if (is_model_NN==True):
            itGd_x_exps= np.asarray(itGd_x_exps)
            iXGd_x_exps= np.asarray(iXGd_x_exps)
            dLif_x_exps= np.asarray(dLif_x_exps)
            lwrp_x_exps= np.asarray(lwrp_x_exps)
            
            itGd_x_exps_mean= np.mean(itGd_x_exps, axis=0)
            iXGd_x_exps_mean= np.mean(iXGd_x_exps, axis=0)
            dLif_x_exps_mean= np.mean(dLif_x_exps, axis=0)
            lwrp_x_exps_mean= np.mean(lwrp_x_exps, axis=0)
            
            itGd_x_exp_ratios= []
            iXGd_x_exp_ratios= []
            dLif_x_exp_ratios= []
            lwrp_x_exp_ratios= []
            

        # ------------------------------------ distance of each explanation from the mean of explanations 
        for j in range(runs):
            t_x_exp_ratios.append(lp_norm_dif(t_x_exps_mean, t_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))

            shap_x_exp_ratios.append(lp_norm_dif(shap_x_exps_mean, shap_x_exps[j], 
                                                 p_norm=descriptor['p_norm'], norm=False))

            lime_x_exp_ratios.append(lp_norm_dif(lime_x_exps_mean, lime_x_exps[j], 
                                                 p_norm=descriptor['p_norm'], norm=False))
            
            if (is_model_NN==True):
                itGd_x_exp_ratios.append(lp_norm_dif(itGd_x_exps_mean, itGd_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                iXGd_x_exp_ratios.append(lp_norm_dif(iXGd_x_exps_mean, iXGd_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                dLif_x_exp_ratios.append(lp_norm_dif(dLif_x_exps_mean, dLif_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                lwrp_x_exp_ratios.append(lp_norm_dif(lwrp_x_exps_mean, lwrp_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
        
            
        # ------------------------------------ max ratio related to each x_data
        t_stability_ratios.append(t_x_exp_ratios[np.argmax(t_x_exp_ratios)])
        shap_stability_ratios.append(shap_x_exp_ratios[np.argmax(shap_x_exp_ratios)])
        lime_stability_ratios.append(lime_x_exp_ratios[np.argmax(lime_x_exp_ratios)])
        
        if (is_model_NN==True):
            itGd_stability_ratios.append(itGd_x_exp_ratios[np.argmax(itGd_x_exp_ratios)])
            iXGd_stability_ratios.append(iXGd_x_exp_ratios[np.argmax(iXGd_x_exp_ratios)])
            dLif_stability_ratios.append(dLif_x_exp_ratios[np.argmax(dLif_x_exp_ratios)])
            lwrp_stability_ratios.append(lwrp_x_exp_ratios[np.argmax(lwrp_x_exp_ratios)])
    
    # ------------------------------------ general max ratio related to all data
    t_max = t_stability_ratios[np.argmax(t_stability_ratios)]
    shap_max= shap_stability_ratios[np.argmax(shap_stability_ratios)]
    lime_max= lime_stability_ratios[np.argmax(lime_stability_ratios)]

    results= {
        't_exp_res': t_max,
        'shap_res': shap_max,
        'lime_res': lime_max,
    }
    
    if (is_model_NN==True):
        itGd_max= itGd_stability_ratios[np.argmax(itGd_stability_ratios)]
        iXGd_max= iXGd_stability_ratios[np.argmax(iXGd_stability_ratios)]
        dLif_max= dLif_stability_ratios[np.argmax(dLif_stability_ratios)]
        lwrp_max= lwrp_stability_ratios[np.argmax(lwrp_stability_ratios)]

        results_grad= {
            'itGd_res': itGd_max,
            'iXGd_res': iXGd_max,
            'dLif_res': dLif_max,
            'lwrp_res': lwrp_max, 
        }
        results.update(results_grad)

    results= {
        key: [float(val) for val in value] if isinstance(value, list) else float(value) for key, value in results.items()
    }

    # max stability_ratios
    return results
    

# Metric -- Prediction Gap on Important Features -- PGI

In [ ]:
def sorted_indices(seq, reverse:bool=False):
    """
    RETURN the indexes of a descending sorted array (seq) if reverse is True, indexes of an 
    ascending sorted array if reverse is False
    """
    seq= np.asarray(seq)
    
    if reverse: seq= -seq
    
    return [i for (v, i) in sorted((v, i) for (i, v) in enumerate(seq))]

In [ ]:
def get_top_k_x_noise(x, e_index, top_k, noise_type='zero'):
    """
    x is a Pandas DataFrame with an instance
    e_index is a vector of indices from an explanation ordered with sorted_indices()
    top_k is an INTEGER representing the number of top features
    x_pert represents the zero/perturbed instance from x used to generate a x'
    TODO: implement different types of noise
    
    RETURN a perturbed instance xi' [i is in top_k]
"""
    
    # get the top k features
    top_k_index= e_index[:top_k]
    names= x.columns

    top_k_names= [names[v] for (i, v) in enumerate(top_k_index)]
    #non_top_k_names= list(filter(lambda x:x not in top_k_names, names))

    x_pert= np.zeros(x.shape)
    x_pert_sample= pd.DataFrame(data=x_pert, columns=x.columns)

    # delete/perturb the top-k important features to produce x'
    x_noise= (x.copy()).reset_index(drop=True)
    x_noise[top_k_names]= x_pert_sample[top_k_names]

    return x_noise

In [ ]:
def eval_pred_faithfulness(model, data, labels, descriptor, top_k=1, noise_type='zero', perturbation=None, 
                           cat_fts=[], train_data=[], labels_train= [], is_model_NN:bool=False):
    """
    Prediction Gap on Important Features
        model is a treined classifier
        data is a preprocessed Pandas DataFrame -- model's training data
        labels are the data labels (Pandas DataFrame)
        perturbation is a OpenXAI perturbation object
        descriptor define the parameters to explanations and data perturbations
        cat_fts list indicating the categorical columns. if empty metric will consider all features as numeric
    
    RETURN PGIF metric considering all features as important for T-Exp, SHAP, and LIME.
    
    The PGIF scores provide insights into the model's predictions, considering both the change in accuracy 
    when the feature is randomized and the difference in Exp values.
    """
    
    top_k= list(np.reshape([top_k], -1))
    top_k= [ int(x) for x in top_k ]     # ensure everything is integer
    
    for i in range(len(top_k)):          # ensure upper and lower boundaries
        if (top_k[i]< 0): top_k[i]= 0
        elif (top_k[i]> data.columns.shape[0]): top_k[i]= data.columns.shape[0]
        
    top_k= list(np.unique(top_k))        # remove repetitions
    
    texp_pgi= [[] for _ in range(len(top_k))]
    shap_pgi= [[] for _ in range(len(top_k))]
    lime_pgi= [[] for _ in range(len(top_k))]

    nn_pytorch_model= None
    if (is_model_NN==True):
        # convert a scikit-learn NN model to a PyTorch NN model used in captum
        nn_pytorch_model= sklearn_to_pytorch_NN(model, data.shape[1])
        
        itGd_pgi= [[] for _ in range(len(top_k))]
        iXGd_pgi= [[] for _ in range(len(top_k))]
        dLif_pgi= [[] for _ in range(len(top_k))]
        lwrp_pgi= [[] for _ in range(len(top_k))]
        
    
    retrained= False
    ohe_model= None
    if (np.asarray(cat_fts).shape[0]> 0):
        # T-Explainer requires a model retraining only for categorical cases
        num_ohe_train_data= ohe_cat_to_numerical_simulator(train_data, cat_fts, 
                                                           delta=descriptor['ohe_delta'], 
                                                           rand_seed=True)
        ohe_model= clone(model)
        ohe_model.fit(num_ohe_train_data, labels_train.values.ravel())
        retrained= True
    
    
    data_size= data.shape[0]
    
    for i_data in tqdm(range(data_size)):
        
        # i_data and its label as pd.DataFrames
        target_x= pd.DataFrame(data=[data.iloc[i_data,:]], columns=data.columns)
        target_y= pd.DataFrame(data=[labels.iloc[i_data]], columns=labels.columns)

        # i_data and its label as tensors
        x_data= torch.tensor(np.asarray(target_x), dtype=torch.float32)
        y_data= torch.tensor(np.asarray(target_y), dtype=torch.float32)
        
        # get the predicted probability of f(x)
        true_label_value= int(y_data.item())
        fx_acc= ML(model, target_x)[true_label_value]
        
        
        # ------------------------------------ get x_data explanation
        x_exps= get_x_explanations(model, data, labels, target_x, target_y, descriptor, cat_fts, 
                                   nn_pytorch_model, is_model_NN, retrained, ohe_model)
        t_x_exp= x_exps['t_exp']
        lime_x_exp= x_exps['lime']
        shap_x_exp= x_exps['shap']
        itGd_x_exp= x_exps['itGd']
        iXGd_x_exp= x_exps['iXGd']
        dLif_x_exp= x_exps['dLif']
        lwrp_x_exp= x_exps['lwrp']

        
        # ------------------------------------ identify the indices of the top-k important features
        """ Magnitude Indicates Importance: The absolute value of the importance score usually reflects 
        the feature's importance. Large negative or positive values both indicate high importance; the 
        sign simply tells you the direction of the feature's contribution to the prediction (positive 
        or negative impact).
        """
        texp_importances= (np.abs(t_x_exp.numpy()))
        texp_importances_index= sorted_indices(texp_importances, reverse=True)
        shap_importances= (np.abs(shap_x_exp.numpy()))
        shap_importances_index= sorted_indices(shap_importances, reverse=True)
        lime_importances= (np.abs(lime_x_exp.numpy()))
        lime_importances_index= sorted_indices(lime_importances, reverse=True)
        
        if (is_model_NN==True):
            itGd_importances= (np.abs(itGd_x_exp.numpy()))
            itGd_importances_index= sorted_indices(itGd_importances, reverse=True)
            iXGd_importances= (np.abs(iXGd_x_exp.numpy()))
            iXGd_importances_index= sorted_indices(iXGd_importances, reverse=True)
            dLif_importances= (np.abs(dLif_x_exp.numpy()))
            dLif_importances_index= sorted_indices(dLif_importances, reverse=True)
            lwrp_importances= (np.abs(lwrp_x_exp.numpy()))
            lwrp_importances_index= sorted_indices(lwrp_importances, reverse=True)
        
        
        # ------------------------------------ iterate for each top_k value of each explanation
        for j in range(len(top_k)):
            top_k_value= top_k[j]

            # delete features that are in the top k ones to produce x'
            texp_target_x= get_top_k_x_noise(target_x, texp_importances_index, top_k_value, noise_type)
            shap_target_x= get_top_k_x_noise(target_x, shap_importances_index, top_k_value, noise_type)
            lime_target_x= get_top_k_x_noise(target_x, lime_importances_index, top_k_value, noise_type)
            
            if (is_model_NN==True):
                itGd_target_x= get_top_k_x_noise(target_x, itGd_importances_index, top_k_value, noise_type)
                iXGd_target_x= get_top_k_x_noise(target_x, iXGd_importances_index, top_k_value, noise_type)
                dLif_target_x= get_top_k_x_noise(target_x, dLif_importances_index, top_k_value, noise_type)
                lwrp_target_x= get_top_k_x_noise(target_x, lwrp_importances_index, top_k_value, noise_type)
        
            # take the difference between f(x) and f(x')
            texp_fx_acc= ML(model, texp_target_x)[true_label_value]
            shap_fx_acc= ML(model, shap_target_x)[true_label_value]
            lime_fx_acc= ML(model, lime_target_x)[true_label_value]
            
            if (is_model_NN==True):
                itGd_fx_acc= ML(model, itGd_target_x)[true_label_value]
                iXGd_fx_acc= ML(model, iXGd_target_x)[true_label_value]
                dLif_fx_acc= ML(model, dLif_target_x)[true_label_value]
                lwrp_fx_acc= ML(model, lwrp_target_x)[true_label_value]

                # take the difference between f(x) and f(x')
                texp_fx_acc_diff= np.abs(fx_acc - texp_fx_acc)
                shap_fx_acc_diff= np.abs(fx_acc - shap_fx_acc)
                lime_fx_acc_diff= np.abs(fx_acc - lime_fx_acc)
                
                if (is_model_NN==True):
                    itGd_fx_acc_diff= np.abs(fx_acc - itGd_fx_acc)
                    iXGd_fx_acc_diff= np.abs(fx_acc - iXGd_fx_acc)
                    dLif_fx_acc_diff= np.abs(fx_acc - dLif_fx_acc)
                    lwrp_fx_acc_diff= np.abs(fx_acc - lwrp_fx_acc)
                    
            
            # compute the mean (1/m)sum(|f(x) - f(x'm)|)
            # high fidelity explanation will result in high accuracy differences when deleting/perturbing
            # the k most important features
            texp_pgi[j].append(np.mean(texp_fx_acc_diff))
            shap_pgi[j].append(np.mean(shap_fx_acc_diff))
            lime_pgi[j].append(np.mean(lime_fx_acc_diff))
            
            if (is_model_NN==True):
                itGd_pgi[j].append(np.mean(itGd_fx_acc_diff))
                iXGd_pgi[j].append(np.mean(iXGd_fx_acc_diff))
                dLif_pgi[j].append(np.mean(dLif_fx_acc_diff))
                lwrp_pgi[j].append(np.mean(lwrp_fx_acc_diff))
          
        
    texp_m_pgi= np.mean(texp_pgi, axis=1)
    texp_m_pgi_sd= np.std(texp_pgi, axis=1)
            
    shap_m_pgi= np.mean(shap_pgi, axis=1)
    shap_m_pgi_sd= np.std(shap_pgi, axis=1)

    lime_m_pgi= np.mean(lime_pgi, axis=1)
    lime_m_pgi_sd= np.std(lime_pgi, axis=1)

    if (is_model_NN==True):
        itGd_m_pgi= np.mean(itGd_pgi, axis=1)
        itGd_m_pgi_sd= np.std(itGd_pgi, axis=1)
        
        iXGd_m_pgi= np.mean(iXGd_pgi, axis=1)
        iXGd_m_pgi_sd= np.std(iXGd_pgi, axis=1)
        
        dLif_m_pgi= np.mean(dLif_pgi, axis=1)
        dLif_m_pgi_sd= np.std(dLif_pgi, axis=1)
        
        lwrp_m_pgi= np.mean(lwrp_pgi, axis=1)
        lwrp_m_pgi_sd= np.std(lwrp_pgi, axis=1)
    
        
    results= {
        'PGI values from top k features': top_k,
        'texp pgi': list(texp_m_pgi),
        'texp pgi std': list(texp_m_pgi_sd),
        'shap pgi': list(shap_m_pgi),
        'shap pgi std': list(shap_m_pgi_sd),
        'lime pgi': list(lime_m_pgi),
        'lime pgi std': list(lime_m_pgi_sd),
    }
        
    if (is_model_NN==True):
        results_grad= {
            'itGd pgi': list(itGd_m_pgi),
            'itGd pgi std': list(itGd_m_pgi_sd),
            'iXGd pgi': list(iXGd_m_pgi),
            'iXGd pgi std': list(iXGd_m_pgi_sd),
            'dLif pgi': list(dLif_m_pgi),
            'dLif pgi std': list(dLif_m_pgi_sd),
            'lwrp pgi': list(lwrp_m_pgi),
            'lwrp pgi std': list(lwrp_m_pgi_sd),
        }
        results.update(results_grad)

    results= {
        key: list(map(int, values)) if isinstance(values[0], np.int64) 
        else list(map(float, values)) if isinstance(values[0], np.float64) 
        else values
            for key, values in results.items()
    }

    return results

# Metric -- Local Accuracy Preservation -- LAP

In [ ]:
def eval_local_accuracy(model, train_data, labels_train, data, labels, descriptor, cat_fts=[], 
                        is_model_NN:bool=False):
    """
    For an additive explanator, local accuracy preservation means that the sum of all feature importance values
    will be equal to the difference between the expected value of the model and the predicted value
    That is, f(x) = phi_0 + Sum(phi_i), with phi_0 = E[f(X)].
    Args:
        model is a treined classifier
        train_data is the Pandas DataFrame with model's training data
        labels_train is the Pandas DataFrame with model's training labels data
        data is a preprocessed Pandas DataFrame to assess
        labels are the data labels (Pandas DataFrame)
        perturbation is a OpenXAI perturbation object
        descriptor define the parameters to explanations and data perturbations
        cat_fts list indicating the categorical columns. if empty metric will consider all features as numeric
    
    RETURN a measure of faithfulness for local accuracy preservation for each explainer over a non-perturbed 
           dataset as a ratio of explanations that preserved local accuracy (mean value, the greater the value,
           the less faithful the method is)
    """

    data_size= data.shape[0]
    # we assume all data are numerical
    mean_inst= replace_values(train_data, train_data.columns, num_type='mean', cat_type='none')
    # e_fx can be understood as the average model output across the training set X when Xi is not known
    # for this reason, the data mean is used
    phi_0= expected_value_x_mean(model, mean_inst, labels_train.values.ravel())
    phi_0_logit= pred_proba_to_log_odds(phi_0)

    if isinstance(model, xgb.XGBModel):
        shap_exp_gen= shap.TreeExplainer(model, train_data, model_output='probability')
    else:
        shap_exp_gen= shap.Explainer(model.predict, train_data)

    shap_x_exp= shap_exp_gen(pd.DataFrame(data=[data.iloc[0,:]], columns=data.columns))
    shap_phi_0= (shap_x_exp.base_values)[0]
    
    texp_lap= 0
    shap_lap= 0
    lime_lap= 0
    
    nn_pytorch_model= None
    if (is_model_NN==True):
        # convert a scikit-learn NN model to a PyTorch NN model used in captum
        nn_pytorch_model= sklearn_to_pytorch_NN(model, train_data.shape[1])
        
        itGd_lap= 0
        iXGd_lap= 0
        dLif_lap= 0
        lwrp_lap= 0
        
        
    retrained= False
    ohe_model= None
    if (np.asarray(cat_fts).shape[0]> 0):
        # T-Explainer requires a model retraining only for categorical cases
        num_ohe_train_data= ohe_cat_to_numerical_simulator(train_data, cat_fts, 
                                                           delta=descriptor['ohe_delta'], 
                                                           rand_seed=True)
        ohe_model= clone(model)
        ohe_model.fit(num_ohe_train_data, labels_train.values.ravel())
        retrained= True
    
    tolerance= descriptor['eps_eval_add']
    
    for i_data in tqdm(range(data_size)):
        
        # i_data and its label as pd.DataFrames
        target_x= pd.DataFrame(data=[data.iloc[i_data,:]], columns=data.columns)
        target_y= pd.DataFrame(data=[labels.iloc[i_data]], columns=labels.columns)
        
        # get the predicted probability of f(x)
        fx_p_class= ((model.predict(target_x)).astype(int))[0]
        fx_p_prob = ML(model, target_x)[fx_p_class]
        fx_p_logit= pred_proba_to_log_odds(fx_p_prob)
        
        fx_tol_pls= fx_p_prob + tolerance
        fx_tol_min= fx_p_prob - tolerance
        
        logit_fx_tol_pls= pred_proba_to_log_odds(fx_tol_pls)
        logit_fx_tol_min= pred_proba_to_log_odds(fx_tol_min)
    
        # ------------------------------------ get x_data explanation        
        x_exps= get_x_explanations(model, data, labels, target_x, target_y, descriptor, cat_fts, 
                                   nn_pytorch_model, is_model_NN, retrained, ohe_model)

        t_x_exp= x_exps['t_exp']
        lime_x_exp= x_exps['lime']
        shap_x_exp= x_exps['shap']
        itGd_x_exp= x_exps['itGd']
        iXGd_x_exp= x_exps['iXGd']
        dLif_x_exp= x_exps['dLif']
        lwrp_x_exp= x_exps['lwrp']
        
        # ------------------------------------ get the additive approximation
        texp_fx= phi_0 + np.sum(t_x_exp.numpy())
        shap_fx= shap_phi_0 + np.sum(shap_x_exp.numpy())
        lime_fx= phi_0 + np.sum(lime_x_exp.numpy())
        
        if (is_model_NN==True):
            itGd_fx= phi_0_logit + np.sum(itGd_x_exp.numpy())
            iXGd_fx= phi_0_logit + np.sum(iXGd_x_exp.numpy())
            dLif_fx= phi_0_logit + np.sum(dLif_x_exp.numpy())
            lwrp_fx= phi_0_logit + np.sum(lwrp_x_exp.numpy())
            
        if (texp_fx>= fx_tol_min and texp_fx<= fx_tol_pls):
            texp_lap= texp_lap + 1
        
        if (shap_fx>= fx_tol_min and shap_fx<= fx_tol_pls):
            shap_lap= shap_lap + 1
        
        if (lime_fx>= fx_tol_min and lime_fx<= fx_tol_pls):
            lime_lap= lime_lap + 1
            
        if (is_model_NN==True):
            if (itGd_fx>= logit_fx_tol_min and itGd_fx<= logit_fx_tol_pls):
                itGd_lap= itGd_lap + 1
                
            if (iXGd_fx>= logit_fx_tol_min and iXGd_fx<= logit_fx_tol_pls):
                iXGd_lap= iXGd_lap + 1
            
            if (dLif_fx>= logit_fx_tol_min and dLif_fx<= logit_fx_tol_pls):
                dLif_lap= dLif_lap + 1
            
            if (lwrp_fx>= logit_fx_tol_min and lwrp_fx<= logit_fx_tol_pls):
                lwrp_lap= lwrp_lap + 1
            
    texp_lap= texp_lap / data_size
    shap_lap= shap_lap / data_size
    lime_lap= lime_lap / data_size
    
    if (is_model_NN==True):
        itGd_lap= itGd_lap / data_size
        iXGd_lap= iXGd_lap / data_size
        dLif_lap= dLif_lap / data_size
        lwrp_lap= lwrp_lap / data_size
    
    results= {
        'texp LAP': texp_lap,
        'shap LAP': shap_lap,
        'lime LAP': lime_lap,
    }
    if (is_model_NN==True):
        results_grad= {
            'itGd LAP': itGd_lap,
            'iXGd LAP': iXGd_lap,
            'dLif LAP': dLif_lap,
            'lwrp LAP': lwrp_lap,
        }
        results.update(results_grad)
    
    return results

# Metric -- Consistency Across Instances -- CAI

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

def explanation_consistency(
    model, train_data, train_labels, target_x, target_y, descriptor, k=5, is_model_NN:bool=False
):
    """
    Compute the average cosine similarity between the explanation of a target instance and the 
    explanations of its k nearest neighbors (restricted to those with the same class label).
    Evaluate how similar the explanations are for similar instances. For example, if two instances 
    are close in feature space, their explanations should also be similar.
    Parameters
    - model: A trained machine learning model
    - train_data (pd.DataFrame): The training data, where each row is an instance.
    - train_labels (pd.DataFrame): The training labels as a single-column DataFrame.
    - target_x (pd.DataFrame): A single-row DataFrame representing the target instance.
    - target_y (pd.DataFrame): A single-row DataFrame (with one column) containing the target's label.
    - descriptor: Dictionary defining parameters.
    - k (int, optional): The number of nearest neighbors to consider (default is 5).
    - is_model_NN (bool, optional): Whether the model is a neural network model
    Returns
    - float: The average cosine similarity between the explanation of the target instance and those of 
    its k nearest neighbors.
        - Higher values (closer to 1) indicate that similar instances receive similar explanations.
        - Lower values suggest that the explanations vary more among similar instances.
    """
    # assume train_labels has one column; get its name and extract labels
    label_col = train_labels.columns[0]
    target_label = target_y.iloc[0, 0]  # get the target's label (assumes one value)
    
    # filter train_data to keep only instances with the same label as the target
    train_data_same = train_data[train_labels[label_col] == target_label]
    
    # convert train_data_same and target_x to numpy arrays
    train_data_same_np = train_data_same.values  # shape: (n_instances, n_features)
    target_x_np = target_x.values  # shape: (1, n_features)
    
    # fit NearestNeighbors on the filtered training data
    nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
    nn.fit(train_data_same_np)
    distances, indices = nn.kneighbors(target_x_np)

    texp_neighbor_exp= []
    shap_neighbor_exp= []
    lime_neighbor_exp= []
    if (is_model_NN==True):
        itGd_neighbor_exp= []
        iXGd_neighbor_exp= []
        dLif_neighbor_exp= []
        lwrp_neighbor_exp= []
        # convert a scikit-learn NN model to a PyTorch NN model used in captum
        nn_pytorch_model= sklearn_to_pytorch_NN(model, train_data.shape[1])
    else:
        nn_pytorch_model= None

    # TODO: fully extend to the categorical texp
    cat_fts=[]
    retrained= False
    ohe_model= None
    
    # ------------------------------------ get target_x explanation
    x_exps= get_x_explanations(
        model, train_data, train_labels, target_x, target_y, descriptor, cat_fts, nn_pytorch_model, 
        is_model_NN, retrained, ohe_model
    )
    texp_x_exp= (x_exps['t_exp']).reshape(1, -1).numpy()
    shap_x_exp= (x_exps['shap']).reshape(1, -1).numpy()
    lime_x_exp= (x_exps['lime']).reshape(1, -1).numpy()
    itGd_x_exp= (x_exps['itGd']).reshape(1, -1).numpy()
    iXGd_x_exp= (x_exps['iXGd']).reshape(1, -1).numpy()
    dLif_x_exp= (x_exps['dLif']).reshape(1, -1).numpy()
    lwrp_x_exp= (x_exps['lwrp']).reshape(1, -1).numpy()
    
    for idx in indices[0]:
        # retrieve neighbor instances (as a DataFrame)
        X_neighbor = train_data_same.iloc[[idx]]

        neighbor_explanations = get_x_explanations(
            model, train_data, train_labels, X_neighbor, target_y, descriptor, cat_fts, nn_pytorch_model, 
            is_model_NN, retrained, ohe_model
        )

        texp_neighbor_exp.append((neighbor_explanations['t_exp']).numpy())
        shap_neighbor_exp.append((neighbor_explanations['shap']).numpy())
        lime_neighbor_exp.append((neighbor_explanations['lime']).numpy())

        if (is_model_NN==True):
            itGd_neighbor_exp.append((neighbor_explanations['itGd']).numpy())
            iXGd_neighbor_exp.append((neighbor_explanations['iXGd']).numpy())
            dLif_neighbor_exp.append((neighbor_explanations['dLif']).numpy())
            lwrp_neighbor_exp.append((neighbor_explanations['lwrp']).numpy())

    # compute cosine similarities (target explanation vs. each neighbor's explanation)
    texp_similarity = cosine_similarity(texp_x_exp, np.asarray(texp_neighbor_exp))
    shap_similarity = cosine_similarity(shap_x_exp, np.asarray(shap_neighbor_exp))
    lime_similarity = cosine_similarity(lime_x_exp, np.asarray(lime_neighbor_exp))
    
    # for a single target, cosine_similarity returns a 1 x k vector.
    results = {
        'texp_cai': np.mean(texp_similarity),
        'shap_cai': np.mean(shap_similarity),
        'lime_cai': np.mean(lime_similarity),
    }

    if is_model_NN:
        itGd_similarity = cosine_similarity(itGd_x_exp, np.asarray(itGd_neighbor_exp))
        iXGd_similarity = cosine_similarity(iXGd_x_exp, np.asarray(iXGd_neighbor_exp))
        dLif_similarity = cosine_similarity(dLif_x_exp, np.asarray(dLif_neighbor_exp))
        lwrp_similarity = cosine_similarity(lwrp_x_exp, np.asarray(lwrp_neighbor_exp))

        results.update({
            'itGd_cai': np.mean(itGd_similarity),
            'iXGd_cai': np.mean(iXGd_similarity),
            'dLif_cai': np.mean(dLif_similarity),
            'lwrp_cai': np.mean(lwrp_similarity)
        })

    return results


In [ ]:
def mean_explanation_consistency(model, train_data, train_labels, descriptor, k=5, is_model_NN=False):
    """
    Compute the mean explanation consistency across all instances in the training data.
    For each instance in the training data (used as the target), this function calls 
    the explanation_consistency method to compute the cosine similarity between the target's explanation 
    and its k nearest neighbors (restricted to those with the same class label). It then returns the 
    average consistency value for each explainer.
    """
    consistency_sums = {}
    count = 0

    # iterate over every instance in train_data
    for idx in tqdm(range(len(train_data))):
        # use the current instance as the target
        x_target = train_data.iloc[[idx]]    # single-row DataFrame
        y_target = train_labels.iloc[[idx]]  # single-row DataFrame
        
        # compute explanation consistency for the target instance
        results = explanation_consistency(model, train_data, train_labels, x_target, y_target, descriptor, k, is_model_NN)
        
        # sum up the consistency values for each explainer metric
        for key, value in results.items():
            consistency_sums[key] = consistency_sums.get(key, 0) + value
        count += 1

    # compute the mean consistency for each explainer metric
    mean_consistency = {key: value / count for key, value in consistency_sums.items()}
    
    return mean_consistency